# Task 3 — Compare the two segmentation methods

This notebook compares the masks from Task 2 with the Task 1 masks. It calculates overlap scores, shows where the methods disagree, and displays examples that were difficult for Task 2.

Task 1 masks are model-made references, not hand-labelled truth. A disagreement does not automatically mean Task 2 is wrong.

**Before running:** run Task 2 and make its `task2_results` folder available, along with the Task 1 `combined_masks` folder. You can set `TASK2_DIR` and `TASK1_MASK_DIR` in the first code cell if the folders are somewhere else.

In [1]:
from pathlib import Path
import os, json, math, warnings
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path.cwd()
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_dir(name, roots):
    for root in roots:
        if root.exists():
            matches = sorted(root.rglob(name))
            for p in matches:
                if p.is_dir(): return p
    return None

DATASET_DIR = Path(os.environ.get("EGG_DATASET_DIR", ROOT / "Dataset"))
TASK1_MASK_DIR = Path(os.environ.get("TASK1_MASK_DIR", "")) if os.environ.get("TASK1_MASK_DIR") else None
TASK2_DIR = Path(os.environ.get("TASK2_DIR", "")) if os.environ.get("TASK2_DIR") else None
if TASK1_MASK_DIR is None:
    TASK1_MASK_DIR = find_dir("combined_masks", [ROOT, Path("/kaggle/input"), Path("/kaggle/working")])
if TASK2_DIR is None:
    TASK2_DIR = find_dir("task2_results", [ROOT, Path("/kaggle/input"), Path("/kaggle/working")])
if TASK1_MASK_DIR is None or TASK2_DIR is None:
    raise FileNotFoundError("Need Task 1 combined_masks and Task 2 task2_results. Set TASK1_MASK_DIR and TASK2_DIR, or attach the results folders.")
PRED_DIR = TASK2_DIR / "pred_masks"
if not PRED_DIR.exists():
    raise FileNotFoundError(f"No pred_masks folder in {TASK2_DIR}; rerun Task 2 and retain its output folder.")
print("Dataset:", DATASET_DIR, "\nTask 1 references:", TASK1_MASK_DIR, "\nTask 2 predictions:", PRED_DIR)


c:\Users\DewangC\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: Need Task 1 combined_masks and Task 2 task2_results. Set TASK1_MASK_DIR and TASK2_DIR, or attach the results folders.

## Find matching images and masks

Images and masks are paired by filename. Images without both a Task 1 reference and a Task 2 prediction are skipped. The table below shows the pairs that were found.

In [ ]:
images = sorted(p for p in DATASET_DIR.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
refs = {p.name[:-len("_mask.png")]: p for p in TASK1_MASK_DIR.rglob("*_mask.png")}
preds = {p.name[:-len("_pred.png")]: p for p in PRED_DIR.rglob("*_pred.png")}
rows, cases = [], []
for image_path in images:
    stem = image_path.stem
    ref_path, pred_path = refs.get(stem), preds.get(stem)
    if ref_path is None or pred_path is None: continue
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    gt = cv2.imread(str(ref_path), cv2.IMREAD_GRAYSCALE) > 127
    pred = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE) > 127
    if gt.shape != image.shape[:2]: gt = cv2.resize(gt.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    if pred.shape != image.shape[:2]: pred = cv2.resize(pred.astype(np.uint8), (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    tp = int((pred & gt).sum()); fp = int((pred & ~gt).sum()); fn = int((~pred & gt).sum())
    eps = 1e-9
    rows.append({"image": image_path.name, "TP": tp, "FP": fp, "FN": fn,
        "IoU": tp/(tp+fp+fn+eps), "Dice": 2*tp/(2*tp+fp+fn+eps),
        "Precision": tp/(tp+fp+eps), "Recall": tp/(tp+fn+eps),
        "GT_pixels": int(gt.sum()), "Pred_pixels": int(pred.sum())})
    cases.append((image_path.name, image, gt, pred))
df = pd.DataFrame(rows)
if df.empty: raise ValueError("No matching image/reference/prediction stems. Check the supplied result folders.")
print(f"Paired {len(df)} images; unmatched images: {len(images)-len(df)}")
display(df.head())


## Score the predictions

IoU and Dice measure how much the masks overlap. Precision shows how much of the predicted area matches the reference; recall shows how much of the reference was found. The bootstrap interval gives a rough sense of uncertainty across images.

In [ ]:
def bootstrap_ci(values, n=5000, seed=42):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    samples = values[rng.integers(0, len(values), (n, len(values)))].mean(axis=1)
    return np.percentile(samples, [2.5, 97.5])
metric_cols = ["IoU", "Dice", "Precision", "Recall"]
summary = pd.DataFrame({"Mean": df[metric_cols].mean(), "Median": df[metric_cols].median(),
                        "Std": df[metric_cols].std(ddof=1)})
summary.loc["IoU", "95% bootstrap CI"] = str(np.round(bootstrap_ci(df.IoU), 4))
display(summary.round(4))
print("IoU < 0.5:", int((df.IoU < .5).sum()), "of", len(df), "| IoU < 0.1:", int((df.IoU < .1).sum()))
display(df.sort_values("IoU").head(10)[["image", "IoU", "Dice", "Precision", "Recall", "FP", "FN"]])


## Look at the results

The plots show the spread of scores and whether predictions tend to cover too much or too little area. In the overlays, green pixels match, red pixels are extra prediction, and blue pixels are reference pixels that the prediction missed.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0,0].hist(df.IoU, bins=np.linspace(0,1,21), color="steelblue", edgecolor="white"); axes[0,0].set(title="Per-image IoU", xlabel="IoU", ylabel="Image count")
axes[0,1].boxplot([df[c] for c in metric_cols], tick_labels=metric_cols); axes[0,1].set(title="Metric spread", ylim=(0,1))
axes[1,0].scatter(df.Recall, df.Precision, alpha=.75); axes[1,0].set(title="Precision vs recall", xlabel="Recall", ylabel="Precision", xlim=(0,1), ylim=(0,1)); axes[1,0].grid(alpha=.2)
axes[1,1].scatter(df.GT_pixels, df.Pred_pixels, alpha=.65); lim=max(df.GT_pixels.max(),df.Pred_pixels.max()); axes[1,1].plot([0,lim],[0,lim],"k--",lw=1); axes[1,1].set(title="Foreground area agreement", xlabel="Task 1 reference pixels", ylabel="Task 2 predicted pixels")
plt.tight_layout(); plt.show()


In [ ]:
worst = set(df.nsmallest(min(6,len(df)), "IoU").image)
selected = [x for x in cases if x[0] in worst]
fig, axes = plt.subplots(len(selected), 4, figsize=(14, 3.3*len(selected)), squeeze=False)
for row, (name, img, gt, pred) in enumerate(selected):
    overlay = img.astype(np.float32).copy()
    for mask, color in [(pred & gt,(0,210,0)),(pred & ~gt,(255,0,0)),(~pred & gt,(0,90,255))]: overlay[mask] = .45*overlay[mask] + .55*np.array(color)
    for col, view, title in [(0,img,"Input"),(1,gt,"Task 1 reference"),(2,pred,"Task 2 prediction"),(3,overlay,"TP green / FP red / FN blue")]:
        axes[row,col].imshow(view, cmap="gray" if col in [1,2] else None); axes[row,col].set_title(title if row else f"{title} — {name[:24]}", fontsize=9); axes[row,col].axis("off")
plt.tight_layout(); plt.show()


## What the results mean and where files go

Low precision usually means the predicted mask includes parts of bowls, hands, shadows, or the background. Low recall means the prediction missed part or all of an egg. Task 1 masks are model-generated references, so this comparison shows agreement with those masks rather than proving which outline is correct.

The per-image scores and summary table are saved as CSV files in 	ask3_analysis by default. Change TASK3_OUT_DIR in the last code cell to choose another folder.


In [ ]:
OUT_DIR = Path(os.environ.get("TASK3_OUT_DIR", ROOT / "task3_analysis")); OUT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_DIR / "per_image_metrics.csv", index=False)
summary.to_csv(OUT_DIR / "metric_summary.csv")
print("Saved CSV reports to", OUT_DIR.resolve())
